# Clustering delle regioni per profilo di mercato del lavoro

Segmentazione delle regioni italiane su più indicatori (occupazione, disoccupazione, inattività, NEET — a seconda di quali sono già stati confermati in `src/config.py` e caricati dal DAG) con K-means, con visualizzazione via PCA.

**Prerequisito**: il DAG deve aver completato almeno un run con successo.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from sqlalchemy import create_engine

from src.config import warehouse_connection_string

engine = create_engine(warehouse_connection_string())

## Costruzione della matrice regioni x indicatori

Media sull'ultimo anno disponibile, per indicatore e per regione (sesso=totale, classe_eta=totale).

In [ ]:
fact = pd.read_sql('SELECT * FROM gold.fact_mercato_lavoro', engine)
dim_territorio = pd.read_sql('SELECT * FROM gold.dim_territorio', engine)
dim_tempo = pd.read_sql('SELECT * FROM gold.dim_tempo', engine)

df = fact.merge(dim_territorio, on='cod_territorio').merge(dim_tempo, on='periodo')
df = df[(df['sesso'] == 'totale') & (df['classe_eta'] == 'totale')]

ultimo_anno = df['anno'].max()
recente = df[df['anno'] == ultimo_anno]

matrice = recente.pivot_table(index='territorio', columns='indicatore', values='valore', aggfunc='mean')
matrice = matrice.dropna()
print(f'Regioni: {len(matrice)}  |  Indicatori disponibili: {list(matrice.columns)}')
matrice.head()

Se qui vedi un solo indicatore, significa che in `src/config.py` è ancora confermato solo `tasso_disoccupazione`: conferma gli altri flow_ref con `python -m src.discover` per un clustering più ricco (il clustering funziona comunque anche con un solo indicatore, ma è meno interessante).

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(matrice.values)

K = min(4, len(matrice) - 1) if len(matrice) > 2 else 2
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
matrice['cluster'] = kmeans.fit_predict(X)

if len(matrice) > K:
    print('Silhouette score:', silhouette_score(X, matrice['cluster']))

## Visualizzazione via PCA (2 componenti)

In [ ]:
pca = PCA(n_components=2)
coords = pca.fit_transform(X)
plot_df = matrice.reset_index()[['territorio', 'cluster']].copy()
plot_df['pc1'], plot_df['pc2'] = coords[:, 0], coords[:, 1]

px.scatter(
    plot_df, x='pc1', y='pc2', color=plot_df['cluster'].astype(str), text='territorio',
    title=f'Cluster regionali per profilo di mercato del lavoro ({ultimo_anno})'
)

## Interpretazione dei cluster: media indicatori per gruppo

In [ ]:
matrice.groupby('cluster').mean(numeric_only=True)

## (Opzionale) Salvare l'assegnazione dei cluster in `gold.cluster_regioni`

per poterla mostrare nella dashboard senza rieseguire il notebook.

In [ ]:
out = matrice.reset_index()[['territorio', 'cluster']]
out['anno_riferimento'] = ultimo_anno

# out.to_sql('cluster_regioni', engine, schema='gold', if_exists='replace', index=False)
out